# AHC VAD — Runtime Cascade (Colab, T4)

Runs the release detector: OpenCV decode (2 fps + frame-diff gate) →
SigLIP ViT-B-16 + trained head → Qwen2.5-VL-3B-Instruct confirmation →
hysteresis merge, then writes a submission JSON that passes `emit.validate`
and prints a score on the public test set.

**Set `DRIVE_HOME` in the config cell to your Drive shortcut path, then Runtime -> Run all.**

Expected layout under `DRIVE_HOME`:
```
AHC/
  solution/        # cascade.py, postprocess.py, emit.py, scorer.py, build_manifest.py, manifest.json
  head.pt          # from colab_train.ipynb
  test/            # videos/*.mp4, videos.csv, ground_truth.csv   (public test kit)
```
Qwen2.5-VL-3B-Instruct is a gated model: if `FROM_PRETRAINED` fails with a permission
error, set `HF_TOKEN` in the config cell. Architecture: fp16 + `attn_implementation="sdpa"` (no bf16, no flash-attn).

In [ ]:
!pip -q install open_clip_torch opencv-python-headless transformers pillow

import torch, os
os.environ.setdefault("HF_TOKEN", "")   # <-- set if Qwen2.5-VL is gated for your HF account
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("torch:", torch.__version__)
print("fp16 supported:", torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 6 if torch.cuda.is_available() else False)

## 1. Copy `solution/`, `head.pt`, `test/` from Drive to local disk

In [ ]:
import shutil, time
from google.colab import drive

drive.mount("/content/drive")

DRIVE_HOME = "/content/drive/MyDrive/AHC"   # <-- EDIT: your Drive shortcut path
WORK = "/content/work"

SRC = {
    "solution": f"{DRIVE_HOME}/solution",
    "head.pt": f"{DRIVE_HOME}/head.pt",
    "test": f"{DRIVE_HOME}/test",
}
for name, path in SRC.items():
    assert os.path.exists(path), f"not found on Drive: {path}"
    dst = os.path.join(WORK, name)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(path, dst)
os.chdir(WORK)
print("work dir:", WORK)
print("files:", sorted(os.listdir(WORK)))

## 2. Build manifest (levels come from ground_truth.csv) and run the cascade

To run on a different set (e.g. the arena's private test), place its manifest as
`manifest.json` here and pass `--videos-dir <dir>` accordingly. `--start/--end` allow
resuming an interrupted run; outputs are merged by re-running the full CLI.

In [ ]:
!python build_manifest.py --dir test --out manifest.json
print("videos in manifest:", os.path.exists("manifest.json") and __import__("json").load(open("manifest.json")).get("videos") and len(__import__("json").load(open("manifest.json"))["videos"]))

In [ ]:
import time
t0 = time.time()
rc = os.system(f"python cascade.py --manifest manifest.json --videos-dir test --out submission.json")
print("exit", rc, "| wall", round(time.time() - t0, 1), "s")

## 3. Validate with emit + score against public ground truth

In [ ]:
import json
from emit import validate
from scorer import score

sub = json.load(open("submission.json", encoding="utf-8"))
manifest = json.load(open("manifest.json", encoding="utf-8"))
errs = validate(sub, [v["video_id"] for v in manifest["videos"]])
print("emit.validate errors:", errs)
print("score:", json.dumps(score("test/ground_truth.csv", "submission.json")))

tot = sum(v["runtime_metadata"]["end_to_end_internal_time_ms"] for v in sub["predictions"])
n = len(sub["predictions"])
print(f"videos={n} reported_ms={tot} avg={tot/n:.0f}ms")

## 4. Copy submission back to Drive

In [ ]:
dst = f"{DRIVE_HOME}/submission.json"
shutil.copy("submission.json", dst)
print("saved:", dst)
print("\nfinal answer:", os.path.join(os.getcwd(), "submission.json"))